# Bedrock Permissions Test

Tests Bedrock API access from this SageMaker notebook.

In [ ]:
import boto3

# Show current identity
sts = boto3.client("sts")
identity = sts.get_caller_identity()
print(f"Account: {identity['Account']}")
print(f"ARN:     {identity['Arn']}")

In [ ]:
# Test Bedrock API access (no model invocation)
bedrock = boto3.client("bedrock", region_name="us-west-2")

# List foundation models - tests bedrock:ListFoundationModels permission
models = bedrock.list_foundation_models(byProvider="Anthropic")
claude_models = [m["modelId"] for m in models["modelSummaries"]]

print(f"Found {len(claude_models)} Anthropic models:")
for m in claude_models[:5]:
    print(f"  - {m}")
if len(claude_models) > 5:
    print(f"  ... and {len(claude_models) - 5} more")

print("\n✓ Bedrock API access working!")

In [ ]:
# Test model invocation (requires Marketplace subscription)
bedrock_runtime = boto3.client("bedrock-runtime", region_name="us-west-2")

try:
    response = bedrock_runtime.converse(
        modelId="us.anthropic.claude-3-5-haiku-20241022-v1:0",
        messages=[{"role": "user", "content": [{"text": "Say OK"}]}],
        inferenceConfig={"maxTokens": 10}
    )
    print(f"Response: {response['output']['message']['content'][0]['text']}")
    print("\n✓ Model invocation working!")
except Exception as e:
    print(f"✗ Model invocation failed: {type(e).__name__}")
    print(f"  {e}")
    print("\n  This usually means Marketplace permissions are needed.")
    print("  Add to your role: aws-marketplace:ViewSubscriptions, aws-marketplace:Subscribe")